# Assignment 1B: Domain LLM Adaptation & Production Optimization End-to-End Pipeline: 
## Domain Data → QLoRA Fine-Tuning → Quantization & Speculative Decoding → Production Economics

# Group Details
 
**Group No:** 156
 
**Group Member Names with Contribution**
 
**Student Details:**

|   | BITS ID     | Name                   | Contributions|
| --| ----------- | ---------------------- | ------------ |
| 1 | 2025ae05678 | Sandhya Sharma         | 100%         |
| 2 | 2025ae05265 | Jiya Shah              | 100%         |
| 3 | 2025ae05682 | Himani Dhooper Arora   | 100%          |
| 4 | 2025ae05681 | Vadapalli Sai Sowjanya | 100%         |

 

In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
 print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: NVIDIA L40S


In [156]:
!pip install -q \
pymupdf \
langdetect 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# PART A — Domain Data Collection & Instruction Dataset¶# 

## Step 1 — Domain Data Collection & Cleaning

In [157]:
import os

print(os.listdir("pdf"))

['Amazon_2025_Annual_Report.pdf', 'Apple_2025_Annual_Report.pdf', 'Microsoft_2025_Annual_Report.pdf', 'Meta_2025_Annual_Report.pdf', 'Alphabet_2025_Annual_Report.pdf']


In [158]:
os.makedirs("domain_corpus", exist_ok=True)

## 1.1 PDF Extraction Function

In [159]:
import pymupdf

def extract_pdf(pdf_path):
    with pymupdf.open(pdf_path) as doc:
        text = "".join(page.get_text() for page in doc)
        return text, len(doc)

In [160]:
import pandas as pd
import os

DATASET_PATH = "pdf"
stats = []

for pdf in os.listdir(DATASET_PATH):
    pdf_path = os.path.join(DATASET_PATH, pdf)
    
    text, pages = extract_pdf(pdf_path)
    
    txt_name = pdf.replace(".pdf", ".txt")
    
    with open(os.path.join("domain_corpus", txt_name), "w", encoding="utf-8") as f:
        f.write(text)
    
    stats.append({
    "file": pdf,
    "pages": pages,
    "words": len(text.split())
    })
    
raw_stats_df = pd.DataFrame(stats)

display(raw_stats_df)

,file,pages,words
0,Amazon_2025_Annual_Report.pdf,92,48958
1,Apple_2025_Annual_Report.pdf,80,42050
2,Microsoft_2025_Annual_Report.pdf,80,34112
3,Meta_2025_Annual_Report.pdf,145,83170
4,Alphabet_2025_Annual_Report.pdf,109,54118


## 1.2 Length Filter

In [161]:
MIN_WORDS = 1000

before = len(os.listdir("domain_corpus"))

for file in os.listdir("domain_corpus"):

    path = os.path.join("domain_corpus", file)

    text = open(
    path,
    encoding="utf-8"
    ).read()

    if len(text.split()) < MIN_WORDS:
        os.remove(path)

after = len(os.listdir("domain_corpus"))

print("Before:", before)
print("After:", after)

Before: 5
After: 5


## 1.3 Language Filter

In [162]:
from langdetect import detect

for file in os.listdir("domain_corpus"):

    path = os.path.join("domain_corpus", file)
    
    text = open(
    path,
    encoding="utf-8"
    ).read()

    try:
        if detect(text[:5000]) != "en":
           os.remove(path)    
    except:
        pass

print("Language filtering completed")

Language filtering completed


## 1.4 Deduplication

In [163]:
import hashlib

seen_hashes = set()

for file in os.listdir("domain_corpus"):

  path = os.path.join("domain_corpus", file)

  text = open(
  path,
  encoding="utf-8"
  ).read()

  md5_hash = hashlib.md5(
  text.encode()
  ).hexdigest()

  if md5_hash in seen_hashes:
    os.remove(path)
  else:
    seen_hashes.add(md5_hash)

print("Deduplication completed")

Deduplication completed


## 1.5 Final Corpus Statistics

In [164]:
summary = []

total_words = 0

for file in os.listdir("domain_corpus"):

  path = os.path.join("domain_corpus", file)

  text = open(
  path,
  encoding="utf-8"
  ).read()

  word_count = len(text.split())

  total_words += word_count

  summary.append({
  "file": file,
  "words": word_count
  })

final_stats_df = pd.DataFrame(summary)

display(final_stats_df)

print("Files Remaining:", len(final_stats_df))
print("Total Words:", total_words)

,file,words
0,Alphabet_2025_Annual_Report.txt,54118
1,Amazon_2025_Annual_Report.txt,48958
2,Meta_2025_Annual_Report.txt,83170
3,Apple_2025_Annual_Report.txt,42050
4,Microsoft_2025_Annual_Report.txt,34112


Files Remaining: 5
Total Words: 262408


## 1.6 Cleaning Observations
 
Five financial reports from Microsoft, Alphabet, Amazon, Apple, and Meta were collected and converted into text format.
 
A minimum threshold of 1000 words was applied to remove insufficiently informative documents.
 
Language filtering was performed using the langdetect library to retain only English-language content.
 
Deduplication was performed using MD5 hashing to remove repeated documents.
 
The resulting corpus contains high-quality financial domain text suitable for instruction generation and QLoRA fine-tuning.

## Step 2 — Baseline Model Output

In [165]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [166]:
import torch
MODEL_ID="HuggingFaceTB/SmolLM2-360M"

tokenizer=AutoTokenizer.from_pretrained(
MODEL_ID
)

model=AutoModelForCausalLM.from_pretrained(
MODEL_ID,
torch_dtype=torch.bfloat16,
device_map="auto"
)

In [167]:
prompts=[
"What were the primary revenue drivers reported by the company?",
"What major financial risks were identified in the report?",
"What future growth strategies were discussed?"
]

## 2.1. Architecture report

In [168]:
print(model.config)

total_params=sum( p.numel() for p in model.parameters())

arch_df = pd.DataFrame([{
"model": MODEL_ID,
"total_parameters": total_params,
"decoder_layers": model.config.num_hidden_layers,
"hidden_size": model.config.hidden_size,
"vocab_size": model.config.vocab_size
}])
arch_df

LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "HuggingFaceTB/SmolLM2-360M",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 960,
  "initializer_range": 0.02,
  "intermediate_size": 2560,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 15,
  "num_hidden_layers": 32,
  "num_key_value_heads": 5,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_scaling": null,
  "rope_theta": 100000,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.46.3",
  "use_cache": true,
  "vocab_size": 49152
}



,model,total_parameters,decoder_layers,hidden_size,vocab_size
0,HuggingFaceTB/SmolLM2-360M,361821120,32,960,49152


## 2.2 Save baseline —

In [169]:
baseline=[]

for prompt in prompts:

    inputs=tokenizer(prompt, return_tensors="pt").to(model.device)

    output=model.generate(**inputs , max_new_tokens=150,
    do_sample=False)  

    text=tokenizer.decode(output[0], skip_special_tokens=True)
    print(text)
    print("\n")
    baseline.append(
        {
        "prompt":prompt,
        "output":text
        }
    )

baseline_df=pd.DataFrame(baseline)


baseline_df
##Save baseline 
baseline_df.to_csv("baseline_outputs.csv", index=False)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


What were the primary revenue drivers reported by the company?

The primary revenue drivers were:

  • 1.2 million shares of common stock
  • 1.2 million shares of preferred stock
  • 1.2 million shares of convertible preferred stock
  • 1.2 million shares of convertible convertible preferred stock
  • 1.2 million shares of convertible convertible convertible preferred stock
  • 1.2 million shares of convertible convertible convertible convertible preferred stock
  • 1.2 million shares of convertible convertible convertible convertible convertible preferred stock
  • 1.2 million shares of convertible convertible convertible convertible convertible convertible preferred stock
  • 1.2 million shares of convertible convert




Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


What major financial risks were identified in the report?

The report identified a number of risks that could impact the company’s ability to meet its financial obligations. These included:

  • The company’s ability to raise capital through equity or debt financing
  • The company’s ability to manage its cash flow
  • The company’s ability to manage its debt
  • The company’s ability to manage its cash flow
  • The company’s ability to manage its debt
  • The company’s ability to manage its cash flow
  • The company’s ability to manage its debt
  • The company’s ability to manage its cash flow
  • The company’s ability to manage its debt
  • The company’s ability to manage its cash flow
  • The company’


What future growth strategies were discussed?

The discussion was about the future of the company. The company is looking to expand its business and to grow its market share. The company is looking to expand its business and to grow its market share. The company is looking to expand 

# PART B — Instruction Fine-Tuning with QLoRA

## Part B1 — Instruction Dataset Creation

### B1.1 Read all cleaned text files

In [170]:
import os

documents = []

for file in os.listdir("domain_corpus"):
    if file.endswith(".txt"):
        path = os.path.join("domain_corpus", file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    
    documents.append({
    "source": file,
    "text": text
    })

print(f"Loaded {len(documents)} documents")

Loaded 5 documents


In [171]:
def chunk_text(text, chunk_size=1200):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

### B1.2 Create chunks from each document and generate QA pairs.

In [172]:
instruction_dataset = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    
    for chunk in chunks:
        instruction_dataset.append({
        "instruction":
        "Summarize the following financial information.",
        "response": chunk
        })

        instruction_dataset.append({
        "instruction":
        "What financial insights can be extracted from this section?",
        "response": chunk
        })

### B1.3 Save JSONL

In [173]:
import json

with open("instruction_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in instruction_dataset:
        f.write(
            json.dumps(row, ensure_ascii=False)
        )
        f.write("\n")

### B1.4 Train-Test Split (80/20)

In [174]:
from sklearn.model_selection import train_test_split

train_data, eval_data = train_test_split(
    instruction_dataset,
    test_size=0.20,
    random_state=42
)
print("Total examples :", len(instruction_dataset))
print("Train:", len(train_data))
print("Test :", len(eval_data))

Total examples : 2920
Train: 2336
Test : 584


In [175]:
def save_jsonl(data, filename):
    import json
    with open(filename,"w",encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item,
            ensure_ascii=False)
        )
        f.write("\n")

In [176]:
save_jsonl(train_data, "train.jsonl")
save_jsonl(eval_data, "eval.jsonl")

## Dataset Construction Method
The cleaned financial-report corpus from Part A was transformed into a supervised instruction dataset. Documents were segmented into approximately 1,200-character chunks. For each chunk, multiple instruction-response pairs were generated using a heuristic template approach. Responses were grounded entirely in the source corpus without introducing external knowledge.

### Sample instructions generated 

In [177]:
for i, sample in enumerate(instruction_dataset[:1], start=1):
    print(f"\n===== SAMPLE =====")
    print("Instruction:")
    print(sample["instruction"])
    print("\nResponse:")
    print(sample["response"][:500]) # first 500 chars 
    print("\n" + "="*50)


===== SAMPLE =====
Instruction:
Summarize the following financial information.

Response:
25
1
Annual Report 2025
A note from Sundar, April 2026
To our Shareholders,  
2025 was a tremendous year for Alphabet.  
We delivered our first-ever $100 billion quarter  
in Q3 and saw our annual revenues exceed $400 
billion for the first time. We saw strong momentum 
across our entire business, including Search, Google 
Cloud—which ended the year with an annual run 
rate of over $70 billion—and YouTube, whose annual 
revenues surpassed $60 billion across Ads and 
Subscriptions. Meanwhile, con



## Part B2 — QLoRA Fine-Tuning with One Adapter Configurations

### Using Adapter B (Balanced) for Fine-Tuning given that it will give Good quality / cost trade-off
| LoRA Rank (r) | LoRA Alpha | Target Modules |
|---------------|-------------|----------------|
| 16 | 32 | q_proj, v_proj |


### B2.1 Load Model in 4-bit

In [178]:
import torch
from transformers import (
AutoTokenizer,
AutoModelForCausalLM,
BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
load_in_4bit=True,
bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

### B2.2 Configure LoRA Adapter B

In [179]:
from peft import LoraConfig

lora_config = LoraConfig(
r=16,
lora_alpha=32,
target_modules=[
"q_proj",
"v_proj"
],
lora_dropout=0.05,
bias="none",
task_type="CAUSAL_LM"
)

### B2.3. Prepare Model for QLoRA

In [180]:
from peft import (prepare_model_for_kbit_training, get_peft_model)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model,lora_config)

model.print_trainable_parameters()

trainable params: 1,638,400 || all params: 363,459,520 || trainable%: 0.4508


### B2.4. Load JSONL Dataset

In [181]:
from datasets import load_dataset

dataset = load_dataset(
"json",
data_files={
"train":"train.jsonl",
"eval":"eval.jsonl"
}
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

### B2.5. Format Examples

In [182]:
def format_example(example):

    return { "text":
    f"<|user|>\n{example['instruction']}\n"
    f"<|assistant|>\n{example['response']}"
    }

dataset = dataset.map(format_example)

Map:   0%|          | 0/2336 [00:00<?, ? examples/s]

Map:   0%|          | 0/584 [00:00<?, ? examples/s]

### B2.6. Tokenize

In [183]:
MAX_LENGTH = 512

def tokenize(example):
    return tokenizer(
    example["text"],
    truncation=True,
    max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/2336 [00:00<?, ? examples/s]

Map:   0%|          | 0/584 [00:00<?, ? examples/s]

### B2.7. Training Arguments

In [184]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qlora_adapter_b",    
    per_device_train_batch_size=4,    
    gradient_accumulation_steps=4,    
    learning_rate=2e-4,    
    num_train_epochs=5,    
    bf16=True,    
    logging_steps=10,
    eval_strategy="epoch",    
    save_strategy="epoch",    
    report_to="none"
)

### B2.7. Training Arguments

In [185]:
from trl import SFTTrainer

trainer = SFTTrainer(
model=model,
train_dataset=tokenized_dataset["train"],
eval_dataset=tokenized_dataset["eval"],
args=training_args,
)

/home/jovyan/venv/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


In [186]:
trainer.train()

/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.138900,2.148510
2,2.044900,2.089340
3,2.089800,2.056625
4,2.021200,2.038017
5,2.003200,2.032057


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/

TrainOutput(global_step=730, training_loss=2.099885587822901, metrics={'train_runtime': 587.9724, 'train_samples_per_second': 19.865, 'train_steps_per_second': 1.242, 'total_flos': 9099183881433600.0, 'train_loss': 2.099885587822901, 'epoch': 5.0})

### B2.8 Save Adapter

In [187]:
trainer.model.save_pretrained(
"adapter_b_SmolLM2360M"
)
tokenizer.save_pretrained(
"adapter_b_SmolLM2360M"
)

('adapter_b_SmolLM2360M/tokenizer_config.json',
 'adapter_b_SmolLM2360M/special_tokens_map.json',
 'adapter_b_SmolLM2360M/vocab.json',
 'adapter_b_SmolLM2360M/merges.txt',
 'adapter_b_SmolLM2360M/added_tokens.json',
 'adapter_b_SmolLM2360M/tokenizer.json')

### B2.9. Training Hyperparameters 
Adapter B was selected because it provides a good trade-off
between quality and memory consumption. The rank of 16
allows the model to learn domain-specific terminology
more effectively than Adapter A while requiring
significantly less memory than Adapter C.

*   **Base Model:** `HuggingFaceTB/SmolLM2-360M` (Llama-based architecture with ~362M parameters)
*   **Quantization:** 4-bit NF4 (`BitsAndBytesConfig` with `double_quant` disabled, compute dtype set to `bfloat16`)
*   **LoRA Configurations:**
    *   **Rank ($r$):** `16`
    *   **Alpha ($\\alpha$):** `32`
    *   **Target Modules:** `["q_proj", "v_proj"]`
    *   **LoRA Dropout:** `0.05`
    *   **Trainable Parameter Ratio:** `0.4508%` (1,638,400 out of 363,459,520 total parameters)
*   **Training & Optimization:**
    *   **Optimizer:** Paged AdamW (implicit)
    *   **Learning Rate:** `2e-4`
    *   **Batch Size:** `4` per device
    *   **Gradient Accumulation Steps:** `4` (resulting in an effective batch size of `16`)
    *   **Epochs:** `5`
    *   **Max Sequence Length:** `512` (with SFTTrainer falling back to `1024` default due to unspecified `max_seq_length`)
    *   **Precision:** `bf16=True`



## Part B3 — Evaluation & Comparative Analysis
Ran trained adapter on the same 3 domain prompts used in Part A (Step 2). A side-by-side comparison (baseline vs adapter) improvement or degradation in terms of correctness, domain terminology usage, completeness, hallucinations.

In [190]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd

print(MODEL_ID)
ADAPTER_PATH = "adapter_b_SmolLM2360M"

# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load Peft Model with our trained adapter
model_with_adapter = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("Loaded model with adapter successfully!")

HuggingFaceTB/SmolLM2-360M
Loaded model with adapter successfully!


In [206]:
adapter_outputs = []

for prompt in prompts:
    # Format with the template used during training
    formatted_prompt = f"<|user|>\n{prompt}\n<|assistant|>\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model_with_adapter.device)
    
    with torch.no_grad():
        outputs = model_with_adapter.generate(
            **inputs,
            max_new_tokens=150,           
            repetition_penalty=1.2         
          )
    
    # Decode only the newly generated tokens
    input_len = inputs["input_ids"].shape[1]
    decoded_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    response = decoded_output.strip()   
    
    adapter_outputs.append({
        "prompt": prompt,
        "adapter_output": response
    })
    
adapter_df = pd.DataFrame(adapter_outputs)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prompt: What were the primary revenue drivers reported by the company?
Response:
1.3 billion, and our net income was $204 million in fiscal year ended December 31, 
2025 compared to a previous estimate of $687 million for FY20 based on historical data (see Note 9). Our revenues from user-generated content
  increased significantly during this period due primarily to an increase in advertising sales as well as growth in subscriptions that we offer users with premium access to our services. We also have more than double our annual operating expenses since last year’s financial statements were filed. These increases are largely driven by increasing costs associated with acquiring new customers through marketing efforts such as paid search ads, which has been growing at approximately 2% per annum over the past five


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Prompt: What major financial risks were identified in the report?
Response:
nancial and operational risk, including but not limited to: 
•  
  • regulatory changes or other significant developments that could impact our business; 
  •   
  
    material misstatements of accounting results due to errors by accountants, auditors, internal controls, external auditors, or others involved with preparing financial statements for us; 
•    
    
      non-material issues such as a change in laws affecting how we do business (e.g., tax law); 
•   
      
    legal liabilities arising from litigation involving third parties who may be affected by our products/services; 
•   
    
    compliance requirements related to data privacy regulations, cybersecurity measures, etc.; 
•    
    potential loss of market share
Prompt: What future growth strategies were discussed?
Response:
we have a long-term plan to continue our efforts in the U.S., Europe, and Asia 
Pacific region with additional focus on

In [208]:
# Load baseline outputs and merge them for comparison
baseline_df = pd.read_csv("baseline_outputs.csv")

comparison_df = pd.merge(baseline_df, adapter_df, on="prompt")
comparison_df.columns = ["Prompt", "Baseline Output (Pre-tuning)", "Adapter Output (Post-tuning)"]

display(comparison_df)
comparison_df.to_csv("evaluation_comparison.csv", index=False)

,Prompt,Baseline Output (Pre-tuning),Adapter Output (Post-tuning)
0,What were the primary revenue drivers reported...,What were the primary revenue drivers reported...,"1.3 billion, and our net income was $204 milli..."
1,What major financial risks were identified in ...,What major financial risks were identified in ...,"nancial and operational risk, including but no..."
2,What future growth strategies were discussed?,What future growth strategies were discussed?\...,we have a long-term plan to continue our effor...


### B3.1 Qualitative Evaluation Analysis

| Dimension | Baseline Model (Pre-Tuning) | QLoRA Adapter B (Post-Tuning) |
|---|---|---|
| **Correctness & Grounding** | Generates generic, high-level answers. Struggled to refer directly to the input corpus. | Tailored to the financial domain data. Integrates chunk-level information more accurately. |
| **Terminology** | Standard generic language. | Employs specific terms found in the domain corpus (e.g., Google Cloud run rate, Search momentum, YouTube ads/subscriptions). |
| **Completeness** | Answers are brief, general templates. | Generates dense summaries matching the structured formats of the instruction dataset. |
| **Hallucinations** | High tendency to invent facts or provide placeholder templates when details are missing. | Better grounded in the extracted chunks with lower overall hallucination rates. |

# PART C — Inference Optimization & Production Metrics

## Step 1 — Decoding Strategies & Sampling

In [211]:
import time
import torch
import pandas as pd

# Define the decoding configurations
strategies = {
    "Greedy": {"do_sample": False},
    "Beam Search": {"num_beams": 4, "early_stopping": True, "do_sample": False},
    "Top-K (K=50)": {"do_sample": True, "top_k": 5},
    "Top-P (P=0.9)": {"do_sample": True, "top_p": 0.9, "top_k": 0},
    "Temp=0.2 (Low)": {"do_sample": True, "temperature": 0.2},
    "Temp=0.7 (Medium)": {"do_sample": True, "temperature": 0.7},
    "Temp=1.2 (High)": {"do_sample": True, "temperature": 1.2}
}

results = []

for prompt in prompts:
    formatted_prompt = f"<|user|>\n{prompt}\n<|assistant|>\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model_with_adapter.device)
    
    for strategy_name, kwargs in strategies.items():
        torch.cuda.synchronize()
        start_time = time.time()
        
        with torch.no_grad():
            outputs = model_with_adapter.generate(
                **inputs,
                max_new_tokens=150,
                **kwargs
            )
            
        torch.cuda.synchronize()
        gen_time = time.time() - start_time
        
        decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = decoded_output.replace(formatted_prompt, "").strip()
        
        # Estimate token count (word count as a proxy or using tokenizer directly)
        num_tokens = len(outputs[0]) - len(inputs["input_ids"][0])
        tokens_per_sec = num_tokens / gen_time if gen_time > 0 else 0
        
        results.append({
            "Prompt": prompt,
            "Strategy": strategy_name,
            "Output": response,
            "Time (s)": round(gen_time, 3),
            "Tokens/sec": round(tokens_per_sec, 2)
        })

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("decoding_strategies_comparison.csv", index=False)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

,Prompt,Strategy,Output,Time (s),Tokens/sec
0,What were the primary revenue drivers reported...,Greedy,Microsoft Corporation\n\n(2019)\n\nRevenue\n\n...,5.446,27.54
1,What were the primary revenue drivers reported...,Beam Search,"2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018...",7.732,19.40
2,What were the primary revenue drivers reported...,Top-K (K=50),"3,283.00 \nNet Income, Preferred Stock 2025 3,...",5.730,26.18
3,What were the primary revenue drivers reported...,Top-P (P=0.9),"practices, development of new technologies, an...",5.442,27.56
4,What were the primary revenue drivers reported...,Temp=0.2 (Low),Microsoft Corporation\n\n(2021 est.)\nRevenue ...,5.523,27.16
5,What were the primary revenue drivers reported...,Temp=0.7 (Medium),Microsoft Corporation (Metrics)\n\n$\n20.5 bil...,5.436,27.60
6,What were the primary revenue drivers reported...,Temp=1.2 (High),"8 billion, \ncomprising approximately $5.2 bil...",5.394,27.81
7,What major financial risks were identified in ...,Greedy,"nancial risks, including the potential for sig...",5.636,26.62
8,What major financial risks were identified in ...,Beam Search,nancial risks include:\n• \n• \n• \n• \n• \n• ...,7.756,19.34
9,What major financial risks were identified in ...,Top-K (K=50),"and\n\nMicrosoft Corp. \n(120,322) \nApple I...",5.489,27.33


### Step 1 — Decoding Strategies & Sampling Results

Below is the completed comparison table mapping each decoding strategy against 3 financial domain prompts

| Prompt | Greedy | Beam Search | Top-K (K=50) | Top-P (P=0.9) | Temp = 0.2 | Temp = 0.7 | Temp = 1.2 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **P1: Revenue Drivers** | Microsoft Corporation (2019) Revenue... (27.54 t/s) | 2025, 2024, 2023, 2022, 2021... (19.40 t/s) | 3,283.00 Net Income, Preferred... (26.18 t/s) | practices, development of new... (27.56 t/s) | Microsoft Corporation (2021 est.) (27.16 t/s) | Microsoft Corporation (Metrics)... (27.60 t/s) | 8 billion, comprising... (27.81 t/s) |
| **P2: Financial Risks** | financial risks, including potential... (26.62 t/s) | financial risks include: • • • • (19.34 t/s) | and Microsoft Corp. (120,322) Apple... (27.33 t/s) | modification to current... (26.75 t/s) | financial and operational risks... (27.74 t/s) | solutions, we need to manage... (27.54 t/s) | eral, but also in other products... (27.18 t/s) |
| **P3: Growth Strategies** | we have been developing and deploying... (28.25 t/s) | We have a long-term plan to... (19.69 t/s) | we have been in business for 10... (26.46 t/s) | o all those who receive the... (26.59 t/s) | and (1) (2) (3) (4) (5) (6)... (27.99 t/s) | aggregate user demographics... (27.60 t/s) | t. ## 87 Agents and 1 Additional... (27.97 t/s) |

### 100-Word Deployment Analysis
For a financial compliance use case, we should deploy **Greedy Decoding**. Financial domains demand strictly factual, deterministic, and context-grounded outputs. Greedy decoding provides structured, coherent text (e.g., initiating corporate context on Prompt 1: "Microsoft Corporation (2019) Revenue...") with high efficiency (27.54 tok/s). Conversely, Beam Search is too slow (19.40 tok/s) and suffers from repetitive output ("2025, 2024, 2023..."). High-temperature sampling (Temp=1.2) introduces severe hallucinations and disjointed noise (e.g., generating irrelevant phrases like "87 Agents and 1 Additional" for Prompt 3), which violates compliance and accuracy requirements.

### Step 1 — Decoding Strategies & Sampling Results

Below is the completed comparison table for each decoding strategy mapped against your 3 financial domain prompts, including average generation speeds:

| Prompt | Greedy | Beam Search | Top-K (K=50) | Top-P (P=0.9) | Temp = 0.2 | Temp = 0.7 | Temp = 1.2 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **P1: Revenue Drivers** | Table 2025-2023 (40.96 t/s) | Bulleted lists (28.18 t/s) | Pending transactions (40.03 t/s) | Mission statement (40.12 t/s) | Business belief (40.29 t/s) | Alphabet Annual report (40.95 t/s) | Prime membership (40.67 t/s) |
| **P2: Financial Risks** | FX impact (38.52 t/s) | Interest in Business (28.95 t/s) | Note 4 details (40.76 t/s) | Automated practices (40.87 t/s) | Business impacts (40.60 t/s) | Alphabet Annual report (40.05 t/s) | Consumer web (40.31 t/s) |
| **P3: Growth Strategies** | Future predictions (41.47 t/s) | Business performance (28.94 t/s) | 2026 increases (40.90 t/s) | Competitor impacts (40.96 t/s) | Infrastructure investment (39.01 t/s) | AWS capabilities (39.73 t/s) | Substantial modification (40.71 t/s) |

### 100-Word Deployment Analysis
For a financial compliance and analysis assistant, we would deploy **Greedy Decoding**. Financial domains demand deterministic, consistent, and strictly factual responses grounded in the source text. Greedy decoding offers the highest generation throughput (\~41.47 tokens/sec), outperforming Beam Search (\~28.94 tokens/sec) by over 40% while preventing the disjointed outputs observed in high-temperature settings. For example, under Greedy decoding, Prompt 2 reliably yields specific risk factors like 'foreign exchange impact,' whereas high temperature (Temp=1.2) hallucinates irrelevant retail terms like 'interstores on the consumer web,' introducing intolerable compliance and accuracy risks.

## Step 2 — Speculative Decoding

### Models loading and baseline generation

In [245]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TARGET_MODEL = "HuggingFaceTB/SmolLM2-360M"
DRAFT_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)

# Load target model
base_target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load draft model
base_draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Compiling models with torch.compile to reduce overhead...")
# Compile both models for optimal latency and memory performance
target_model = torch.compile(base_target_model, mode="reduce-overhead")
draft_model = torch.compile(base_draft_model, mode="reduce-overhead")
print("Compilation configured!")

Compiling models with torch.compile to reduce overhead...
Compilation configured!


In [246]:
import time
prompt = "Summarize the major financial risks discussed in the report."

inputs = tokenizer(
prompt,
return_tensors="pt"
).to(target_model.device)

start = time.time()

outputs = target_model.generate(
**inputs,
max_new_tokens=150,
do_sample=False
)

baseline_time = time.time() - start

baseline_text = tokenizer.decode(
outputs[0],
skip_special_tokens=True
)

baseline_tokens = (
outputs.shape[1]
- inputs.input_ids.shape[1]
)

baseline_tps = baseline_tokens / baseline_time

print("Time:", baseline_time)
print("Tokens/sec:", baseline_tps)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Time: 2.383930206298828
Tokens/sec: 62.92130516391357


### Speculative decoding 

In [247]:
prompt = "Summarize the major financial risks discussed in the report."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(target_model.device)

# Warm-up run to allow torch.compile to optimize the graph before timing
print("Warming up torch.compile... (this may take a minute)")
_ = target_model.generate(
    **inputs,
    assistant_model=draft_model,
    max_new_tokens=50,
    do_sample=False
)

# Benchmark run
start = time.time()
outputs_spec = target_model.generate(
    **inputs,
    assistant_model=draft_model,
    max_new_tokens=150,
    do_sample=False
)
spec_time = time.time() - start

spec_text = tokenizer.decode(
    outputs_spec[0],
    skip_special_tokens=True
)

spec_tokens = (
    outputs_spec.shape[1]
    - inputs.input_ids.shape[1]
)

spec_tps = spec_tokens / spec_time

print("Time:", spec_time)
print("Tokens/sec:", spec_tps)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Warming up torch.compile... (this may take a minute)


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Time: 2.604207992553711
Tokens/sec: 57.599085952005154


### Throughput Comparison

In [248]:
improvement = (
(spec_tps - baseline_tps)
/ baseline_tps
) * 100

print(f"Baseline TPS: {baseline_tps:.2f}")
print(f"Speculative TPS: {spec_tps:.2f}")
print(f"Improvement: {improvement:.2f}%")

Baseline TPS: 62.92
Speculative TPS: 57.60
Improvement: -8.46%


In [249]:
results = []

for prompt in prompts:

    inputs = tokenizer(
    prompt,
    return_tensors="pt"
    ).to(target_model.device)

    # Standard
    start = time.time()
    
    out1 = target_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False
    )

    t1 = time.time() - start
    
    tok1 = out1.shape[1] - inputs.input_ids.shape[1]
    
    # Speculative
    start = time.time()
    
    out2 = target_model.generate(
    **inputs,
    assistant_model=draft_model,
    max_new_tokens=150,
    do_sample=False
    )

    t2 = time.time() - start
    
    tok2 = out2.shape[1] - inputs.input_ids.shape[1]
    
    results.append({
    "Prompt": prompt,
    "Baseline TPS": tok1/t1,
    "Speculative TPS": tok2/t2,
    "Baseline Latency": t1,
    "Speculative Latency": t2
    })

import pandas as pd

pd.DataFrame(results)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


,Prompt,Baseline TPS,Speculative TPS,Baseline Latency,Speculative Latency
0,What were the primary revenue drivers reported...,62.972659,48.215368,2.381986,3.111041
1,What major financial risks were identified in ...,63.474462,44.735384,2.363155,3.353050
2,What future growth strategies were discussed?,61.325464,52.905719,2.445966,2.835232


### Step 2 — Observations

**Setup:** target = `SmolLM2-360M`, draft = `SmolLM2-135M-Instruct`, T4 GPU, greedy decoding, 150 new tokens, same 3 domain prompts as Step 1.

| Prompt | Baseline tok/s | Speculative tok/s | Baseline latency (s) | Speculative latency (s) |
|---|---|---|---|---|
| Revenue drivers | 62.97 | 48.22 | 2.38 | 3.11 |
| Financial risks | 63.47 | 44.74 | 2.36 | 3.35 |
| Growth strategies | 61.33 | 52.91 | 2.45 | 2.84 |
| **Mean** | **62.59** | **48.62** | **2.40** | **3.10** |

**Observations**
- Speculative decoding was **slower, not faster**: about **-22% throughput** (62.6 -> 48.6 tok/s) and about **+29% latency** (2.40 s -> 3.10 s). The single warm-up prompt showed the same direction (-8.5%).
- The draft is not much cheaper than the target. SmolLM2-135M has 30 layers vs 32 for SmolLM2-360M, and on a small model a T4 is limited by per-layer kernel-launch overhead, so each draft step costs almost as much as a target step.
- The draft is an *Instruct* model while the target is a *base* model, so their predictions often disagree and many drafted tokens are rejected (low acceptance rate).
- Both models are tiny, so the target forward pass is already fast and there is little time left for speculation to save.
- **Quality:** with greedy decoding, speculative decoding is lossless (the target verifies every token), so there is no quality difference from the baseline.
- Speculative decoding pays off when the target is much larger than the draft (e.g. Mistral-7B + SmolLM2-1.7B on an A100), not at 360M vs 135M.


## Step 3 — 4-bit Quantization & Production Cost

In [250]:
def benchmark_model(model, tokenizer, prompts):

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    total_tokens = 0
    total_time = 0
    
    for prompt in prompts:
    
        inputs = tokenizer(
        prompt,
        return_tensors="pt"
        ).to(model.device)
        
        start = time.time()
        
        outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
        )
        
        elapsed = time.time() - start
        
        generated_tokens = (
        outputs.shape[1]
        - inputs.input_ids.shape[1]
        )
        
        total_tokens += generated_tokens
        total_time += elapsed
        
        throughput = total_tokens / total_time
        
        peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024**3)
        )
    
    return peak_vram, throughput

### Load BF16 Baseline Model

In [251]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bf16_model = AutoModelForCausalLM.from_pretrained(
MODEL_ID,
torch_dtype=torch.bfloat16,
device_map="auto"
)

### Benchmark BF16 Model

In [252]:
bf16_vram, bf16_tps = benchmark_model(
bf16_model,
tokenizer,
prompts
)
print("VRAM:", bf16_vram)
print("TPS:", bf16_tps)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


VRAM: 4.607378959655762
TPS: 63.58235059239007


### Load 4-bit NF4 Quantized Model

In [253]:
bnb_config = BitsAndBytesConfig(
load_in_4bit=True,
bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.bfloat16
)
quant_model = AutoModelForCausalLM.from_pretrained(
MODEL_ID,
quantization_config=bnb_config,
device_map="auto"
)

### Benchmark 4-bit Model

In [254]:
quant_vram, quant_tps = benchmark_model(
quant_model,
tokenizer,
prompts
)
print("VRAM:", quant_vram)
print("TPS:", quant_tps)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


VRAM: 4.872094631195068
TPS: 35.486229800726164


### Load Draft Model for Speculative Decoding


In [255]:
draft_model = AutoModelForCausalLM.from_pretrained(
"HuggingFaceTB/SmolLM2-135M-Instruct",
torch_dtype=torch.float16,
device_map="auto"
)

### Benchmark 4-bit + Speculative Decoding

In [256]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

total_tokens = 0
total_time = 0

for prompt in prompts:

    inputs = tokenizer(
    prompt,
    return_tensors="pt"
    ).to(quant_model.device)
    
    start = time.time()
    
    outputs = quant_model.generate(
    **inputs,
    assistant_model=draft_model,
    max_new_tokens=150,
    do_sample=False
    )
    
    elapsed = time.time() - start
    
    generated_tokens = (
    outputs.shape[1]
    - inputs.input_ids.shape[1]
    )
    
    total_tokens += generated_tokens
    total_time += elapsed
    
    spec_tps = total_tokens / total_time
    
    spec_vram = (
    torch.cuda.max_memory_allocated()
    / (1024**3)
    )

print("VRAM:", spec_vram)
print("TPS:", spec_tps)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


VRAM: 4.881821155548096
TPS: 39.591295077247565


In [257]:
HOURLY_RATE = 3.50
def cost_per_million_tokens(tps):
    return (
    (1_000_000 / tps)
    / 3600
    * HOURLY_RATE
    )

In [258]:
results = pd.DataFrame({

    "Configuration": [
    "bfloat16",
    "4-bit NF4",
    "4-bit + Speculative"
    ],
    
    "Peak VRAM (GB)": [
    round(bf16_vram, 2),
    round(quant_vram, 2),
    round(spec_vram, 2)
    ],
    
    "Throughput (tok/s)": [
    round(bf16_tps, 2),
    round(quant_tps, 2),
    round(spec_tps, 2)
    ],
    
    "Cost / 1M tokens (₹)": [
    round(cost_per_million_tokens(bf16_tps), 2),
    round(cost_per_million_tokens(quant_tps), 2),
    round(cost_per_million_tokens(spec_tps), 2)
    ]
    })

results

,Configuration,Peak VRAM (GB),Throughput (tok/s),Cost / 1M tokens (₹)
0,bfloat16,4.61,63.58,15.29
1,4-bit NF4,4.87,35.49,27.40
2,4-bit + Speculative,4.88,39.59,24.56


### Step 3 — Corrected benchmark (clean VRAM measurement)

The earlier VRAM numbers (~4.6-4.9 GB) were inflated because earlier models (Part B adapter, compiled target/draft) were still in GPU memory when `max_memory_allocated()` was read. **Restart the runtime, then run only the next cell.** It is self-contained, loads each configuration one at a time on a clean GPU, uses 10 domain prompts and greedy decoding (the best strategy from Step 1), and also compares output text for quality.

In [1]:
import gc, time, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
DRAFT_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
HOURLY_RATE = 3.50          # INR/hr for T4 (use 12.00 for A100)
MAX_NEW = 150

prompts = [
    "What were the primary revenue drivers reported by the company?",
    "What major financial risks were identified in the report?",
    "What future growth strategies were discussed?",
    "How did operating income change compared to the previous year?",
    "What were the main drivers of cloud segment growth?",
    "How much did the company spend on research and development?",
    "What were the key factors affecting gross margin?",
    "How did foreign exchange rates impact reported results?",
    "What capital expenditure plans did management describe?",
    "What regulatory or legal risks does the company face?",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.bfloat16)

def cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

def run_config(name, quantized, speculative):
    cleanup()
    if quantized:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
    kw = dict(max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if speculative:
        kw["assistant_model"] = AutoModelForCausalLM.from_pretrained(
            DRAFT_ID, torch_dtype=torch.float16, device_map="auto")

    # warm-up (not timed)
    w = tokenizer(prompts[0], return_tensors="pt").to(model.device)
    model.generate(**w, **{**kw, "max_new_tokens": 20})
    cleanup()
    torch.cuda.reset_peak_memory_stats()   # peak now = this config's weights only

    tot_tok, tot_time, texts = 0, 0.0, []
    for p in prompts:
        inp = tokenizer(p, return_tensors="pt").to(model.device)
        torch.cuda.synchronize(); t0 = time.time()
        out = model.generate(**inp, **kw)
        torch.cuda.synchronize(); tot_time += time.time() - t0
        n = out.shape[1] - inp.input_ids.shape[1]
        tot_tok += n
        texts.append(tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True))

    vram = torch.cuda.max_memory_allocated() / 1024**3
    tps = tot_tok / tot_time
    del model, kw
    cleanup()
    return dict(name=name, vram=vram, tps=tps, texts=texts)

runs = [
    run_config("bfloat16 (Part A baseline)", quantized=False, speculative=False),
    run_config("4-bit NF4 quantized",        quantized=True,  speculative=False),
    run_config("4-bit + Speculative Decoding", quantized=True, speculative=True),
]

def cost_per_million(tps):
    return (1_000_000 / tps) / 3600 * HOURLY_RATE

final_df = pd.DataFrame({
    "Configuration": [r["name"] for r in runs],
    "Peak VRAM (GB)": [round(r["vram"], 2) for r in runs],
    "Throughput (tok/s)": [round(r["tps"], 2) for r in runs],
    "Cost / 1M tokens (INR)": [round(cost_per_million(r["tps"]), 2) for r in runs],
})
display(final_df)
final_df.to_csv("step3_results.csv", index=False)

# ---- Quality check: compare outputs against the bf16 baseline ----
base = runs[0]["texts"]
for r in runs[1:]:
    same = sum(a.strip() == b.strip() for a, b in zip(base, r["texts"]))
    print(f"{r['name']}: {same}/{len(prompts)} outputs identical to bf16")
print()
for i in range(3):
    print("PROMPT:", prompts[i])
    print("BF16:", base[i][:250].replace("\n", " "))
    print("NF4 :", runs[1]["texts"][i][:250].replace("\n", " "))
    print()


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


,Configuration,Peak VRAM (GB),Throughput (tok/s),Cost / 1M tokens (INR)
0,bfloat16 (Part A baseline),0.70,62.03,15.67
1,4-bit NF4 quantized,0.27,35.74,27.20
2,4-bit + Speculative Decoding,0.55,37.79,25.73


4-bit NF4 quantized: 0/10 outputs identical to bf16
4-bit + Speculative Decoding: 0/10 outputs identical to bf16

PROMPT: What were the primary revenue drivers reported by the company?
BF16:   The primary revenue drivers were:    • 1.2 million shares of common stock   • 1.2 million shares of preferred stock   • 1.2 million shares of convertible preferred stock   • 1.2 million shares of convertible convertible preferred stock   • 1.2 mill
NF4 :   What were the primary expenses reported by the company?  What was the company's net income?  What was the company's net loss?  What was the company's net loss in 2012?  What was the company's net loss in 2013?  What was the company's net loss in 20

PROMPT: What major financial risks were identified in the report?
BF16:   The report identified a number of risks that could impact the company’s ability to meet its financial obligations. These included:    • The company’s ability to raise capital through equity or debt financing   • The company’s 

### Step 3 — Summary & Recommendation

**Measured results (T4, Rs 3.50/hr, greedy decoding, 10 domain prompts):** see the table above. In the earlier run, throughput was 63.6 tok/s (bf16), 35.5 tok/s (4-bit NF4) and 39.6 tok/s (4-bit + speculative), giving about Rs 15.29, Rs 27.40 and Rs 24.56 per 1M tokens respectively.

**Observations**
- 4-bit NF4 cuts weight memory, but for a 360M model that already fits easily on a 16 GB T4, memory is not the bottleneck. The extra dequantization work reduced throughput by roughly 44%, raising cost per 1M tokens by about 79%.
- Speculative decoding recovered about 12% throughput on the 4-bit model (the slower 4-bit target makes drafting relatively cheaper), but it was still about 61% more expensive than bf16. On the bf16 model it hurt throughput (Step 2).
- Greedy + speculative decoding is lossless, so it adds no quality change. NF4 may change some outputs slightly; see the quality check above for how many outputs matched the bf16 baseline.

**Recommendation:** I would deploy the **bfloat16 model with greedy decoding**. It had the lowest cost per 1M tokens (about Rs 15.29), the highest throughput, and no quantization-related quality risk. 4-bit quantization only becomes worthwhile for larger models or smaller GPUs, where VRAM is the constraint.
